# GGCL Open Set Recognition (OSR) Benchmark on Kaggle

This notebook runs the complete Open Set Recognition pipeline for **GGCL**:
1. **Houston Cross-Scene** (Houston-13 $\rightarrow$ Houston-18)
2. **HyRank Cross-Scene** (Dioni $\rightarrow$ Loukia)

**GPU accelerator**: Make sure GPU is enabled in the sidebar (**Settings** $\rightarrow$ **Accelerator** $\rightarrow$ **GPU T4 x2** or **P100**).

In [ ]:
# Step 0: Verify GPU Availability
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 1: Workspace Setup & Dataset Auto-Detection / Extraction
import os, shutil, glob, zipfile

working_dir = "/kaggle/working"
input_dir = "/kaggle/input"

print("=== Checking /kaggle/input ===")
for item in glob.glob("/kaggle/input/**/*", recursive=False):
    print(" -", item)

# 1. Ensure GGCL-Open-Set exists in working directory
target_dir = os.path.join(working_dir, "GGCL-Open-Set")
if not os.path.exists(target_dir):
    for root, dirs, files in os.walk(input_dir):
        if "cross-scene" in dirs and "hyrank" in dirs:
            print(f"Copying code from {root} to {target_dir}...")
            shutil.copytree(root, target_dir)
            break

os.makedirs(os.path.join(target_dir, "cross-scene", "data"), exist_ok=True)
os.makedirs(os.path.join(target_dir, "hyrank", "data"), exist_ok=True)

# 2. Auto-detect and extract dataset zips if present
cs_data_dir = os.path.join(target_dir, "cross-scene", "data")
hr_data_dir = os.path.join(target_dir, "hyrank", "data")

def auto_extract_dataset(target_name, dest_dir):
    for search_root in [input_dir, working_dir]:
        if not os.path.isdir(search_root):
            continue
        for root, dirs, files in os.walk(search_root):
            # Loose file check
            for f in files:
                if f.lower() == target_name.lower():
                    src_f = os.path.join(root, f)
                    dst_f = os.path.join(dest_dir, f)
                    if not os.path.exists(dst_f):
                        print(f"Copying {src_f} -> {dst_f}")
                        shutil.copy2(src_f, dst_f)
                # Zip check
                elif f.lower().endswith('.zip'):
                    zpath = os.path.join(root, f)
                    try:
                        with zipfile.ZipFile(zpath, 'r') as zf:
                            names = [n.lower() for n in zf.namelist()]
                            if any(target_name.lower() in n for n in names):
                                print(f"⚡ Auto-extracting {zpath} -> {dest_dir}")
                                zf.extractall(dest_dir)
                    except Exception:
                        pass

auto_extract_dataset('Houston13.mat', cs_data_dir)
auto_extract_dataset('Dioni.mat', hr_data_dir)

print("\n=== Dataset Status ===")
h13_found = os.path.exists(os.path.join(cs_data_dir, 'Houston13.mat'))
dioni_found = os.path.exists(os.path.join(hr_data_dir, 'Dioni.mat'))
print(f"Houston Cross-Scene Data Ready: {h13_found} ({cs_data_dir})")
print(f"HyRank Cross-Scene Data Ready:  {dioni_found} ({hr_data_dir})")
if not h13_found:
    print("⚠️ Notice: If Houston13.mat is not ready, click '+ Add Input' in sidebar to attach 'Houston-cross-scene-data.zip'!")

---
## 2. Houston Cross-Scene (Houston 2013 $\rightarrow$ Houston 2018)
Runs:
- **Step 1**: Pretrain HSINet on H13 + 64-dim Feature Extraction (GPU)
- **Step 2**: Closed-Set Few-Shot SVM baseline
- **Step 3**: Open Set Recognition (OSR) evaluation (MSP novelty scoring, Macro AUROC & Known Accuracy across 20 splits and 5 seeds)

In [ ]:
%cd /kaggle/working/GGCL-Open-Set/cross-scene
!python run.py

In [ ]:
# Display Houston OSR Results
res_path = "CrossScene/CrossScene_OSR_results.txt"
if os.path.exists(res_path):
    print(open(res_path).read())
else:
    print("Results file not found yet.")

---
## 3. HyRank Cross-Scene (Dioni $\rightarrow$ Loukia)
Runs:
- **Step 1**: Pretrain HSINet on Dioni + 64-dim Feature Extraction (GPU)
- **Step 2**: Closed-Set Few-Shot SVM baseline
- **Step 3**: Open Set Recognition (OSR) evaluation (MSP novelty scoring, Macro AUROC & Known Accuracy across 84 splits and 5 seeds)

In [ ]:
%cd /kaggle/working/GGCL-Open-Set/hyrank
!python run.py

In [ ]:
# Display HyRank OSR Results
res_path = "HyRank/HyRank_OSR_results.txt"
if os.path.exists(res_path):
    print(open(res_path).read())
else:
    print("Results file not found yet.")

---
## 4. Package Results for Download

In [ ]:
%cd /kaggle/working
!mkdir -p OSR_Results_Summary_GGCL
!cp -f GGCL-Open-Set/cross-scene/CrossScene/CrossScene_OSR_results.txt OSR_Results_Summary_GGCL/ || true
!cp -f GGCL-Open-Set/cross-scene/CrossScene/CrossScene_results.txt OSR_Results_Summary_GGCL/ || true
!cp -f GGCL-Open-Set/hyrank/HyRank/HyRank_OSR_results.txt OSR_Results_Summary_GGCL/ || true
!cp -f GGCL-Open-Set/hyrank/HyRank/HyRank_results.txt OSR_Results_Summary_GGCL/ || true
!zip -r OSR_Results_Summary_GGCL.zip OSR_Results_Summary_GGCL/
print("Results packaged in /kaggle/working/OSR_Results_Summary_GGCL.zip")